# MultiHead Model Training
### Multi-Task Ordinal Regression for MSK Wrist Ultrasound Pathology Scoring

This notebook implements the **Hydra architecture**: a single unified PyTorch model with:
- A **swappable foundation-model backbone** (RadImageNet / OpenUS / EfficientNet)
- **Metadata conditioning** via learned embeddings (modality + joint type)
- **Modality-gated task heads** — structural branch (B-Mode) + vascular branch (Doppler)
- **Ordinal Binary Decomposition** for 0–3 clinical scores
- **Masked multi-task loss** that strictly ignores NaN / impossible labels
- **Hospital-stratified GroupKFold** CV with Hospital B held out as a blind test set

Connects to outputs from `01_preprocessing` (cropped PNGs) and `02_data_cleaning` (clean DataFrame).

## Cell 1 — Imports & Configuration

In [ ]:
# ─── Standard Library ────────────────────────────────────────────────────────
import os
import copy
import json
import logging
import random
import warnings
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

# ─── Numerical / Data ────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.stats import spearmanr

# ─── Scikit-Learn ────────────────────────────────────────────────────────────
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold
from sklearn.metrics import cohen_kappa_score, confusion_matrix

# ─── PyTorch Core ────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler

# ─── Vision ──────────────────────────────────────────────────────────────────
from PIL import Image
import torchvision.models as tvm
from torchvision.models import (
    ResNet50_Weights,
    DenseNet121_Weights,
    EfficientNet_B2_Weights,
)

# ─── Augmentation ────────────────────────────────────────────────────────────
import albumentations as A
from albumentations.pytorch import ToTensorV2

warnings.filterwarnings('ignore', category=UserWarning)
logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
log = logging.getLogger('hydra')

# ─────────────────────────────────────────────────────────────────────────────
# REPRODUCIBILITY SEED
# ─────────────────────────────────────────────────────────────────────────────
def seed_everything(seed: int = 42):
    """Pin all RNG sources so results are reproducible across runs."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ─────────────────────────────────────────────────────────────────────────────
# DEVICE
# ─────────────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
log.info(f'Using device: {DEVICE}')
if DEVICE.type == 'cuda':
    log.info(f'  GPU: {torch.cuda.get_device_name(0)}')
    log.info(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ─────────────────────────────────────────────────────────────────────────────
# CENTRAL CONFIGURATION DATACLASS
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class HydraConfig:
    """Single source of truth for every hyperparameter."""
    # ── Paths ─────────────────────────────────────────────────────────────
    IMAGE_DIR: str           = 'data/cropped_images'
    LABELS_CSV: str          = 'data/labels_clean.csv'
    CHECKPOINT_DIR: str      = 'checkpoints/hydra'
    RESULTS_DIR: str         = 'results/hydra'
    HOSPITAL_B_ECO_IDS: List[str] = field(default_factory=list)

    # ── Backbone ──────────────────────────────────────────────────────────
    BACKBONE_NAME: str           = 'radimagenet_resnet50'
    BACKBONE_WEIGHTS_PATH: Optional[str] = None
    FREEZE_BACKBONE_EPOCHS: int  = 5

    # ── Architecture ──────────────────────────────────────────────────────
    FEATURE_DIM: int         = 512
    MODALITY_EMBED_DIM: int  = 16
    JOINT_EMBED_DIM: int     = 32
    NUM_JOINT_TYPES: int     = 6
    NUM_MODALITIES: int      = 2
    DROPOUT_RATE: float      = 0.40

    # ── Task Definitions ──────────────────────────────────────────────────
    TASKS: Dict = field(default_factory=lambda: {
        'eg_sinovial':          {'branch': 'structural', 'col': 'eg_sinovial',  'weight': 1.0},
        'bone_erosion':         {'branch': 'structural', 'col': 'bone_erosion', 'weight': 1.5},
        'extensor_hypertrophy': {'branch': 'structural', 'col': 'eg_ext_comun', 'weight': 1.2},
        'pd_sinovial':          {'branch': 'vascular',   'col': 'pd_sinovial',  'weight': 1.0},
        'extensor_doppler':     {'branch': 'vascular',   'col': 'pd_ext_comun', 'weight': 1.2},
    })
    ORDINAL_MAX_SCORE: int   = 3

    @property
    def ORDINAL_THRESHOLDS(self) -> int:
        return self.ORDINAL_MAX_SCORE

    # ── Image / Augmentation ──────────────────────────────────────────────
    IMAGE_SIZE: int          = 320
    GRAYSCALE_TO_RGB: bool   = True

    # ── Training ──────────────────────────────────────────────────────────
    BATCH_SIZE: int          = 24
    NUM_EPOCHS: int          = 60
    LEARNING_RATE: float     = 3e-4
    BACKBONE_LR_MULTIPLIER: float = 0.05
    WEIGHT_DECAY: float      = 1e-4
    NUM_WORKERS: int         = 4
    PIN_MEMORY: bool         = True
    USE_AMP: bool            = True
    GRAD_CLIP_NORM: float    = 1.0
    LR_T0: int               = 20
    LR_T_MULT: int           = 2
    EARLY_STOP_PATIENCE: int = 15

    # ── Cross-Validation ──────────────────────────────────────────────────
    N_FOLDS: int             = 5
    RANDOM_SEED: int         = 42
    PATIENT_ID_COL: str      = 'patient_id'
    HOSPITAL_COL: str        = 'hospital_id'
    ECO_ID_COL: str          = 'eco_id'
    MODALITY_COL: str        = 'tipo_imagen_encoded'
    JOINT_TYPE_COL: str      = 'joint_type_encoded'

CFG = HydraConfig()
seed_everything(CFG.RANDOM_SEED)
Path(CFG.CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
Path(CFG.RESULTS_DIR).mkdir(parents=True, exist_ok=True)
log.info('Configuration loaded.')
log.info(f'  Backbone  : {CFG.BACKBONE_NAME}')
log.info(f'  Image size: {CFG.IMAGE_SIZE}x{CFG.IMAGE_SIZE}')
log.info(f'  Tasks     : {list(CFG.TASKS.keys())}')
log.info(f'  Device    : {DEVICE}')

## Cell 2 — Backbone Factory

Centralises all backbone loading logic. New foundation models can be added here
without touching the Hydra model class.

**Design decisions:**
- Every backbone is stripped of its classification head and returns a flat
  feature vector of shape `(B, backbone_out_dim)`.
- A `nn.Linear` projection then maps to `CFG.FEATURE_DIM` for the task heads.
- RadImageNet weights are loaded from a local `.pth` file.
- OpenUS ViT weights follow the same pattern.
- EchoCare / US-JEPA (2026): stubs provided — fill in once weights are public.

In [ ]:
class FeatureExtractor(nn.Module):
    """Wraps any backbone, strips its head, and exposes .out_dim."""
    def __init__(self, backbone: nn.Module, out_dim: int):
        super().__init__()
        self.backbone = backbone
        self.out_dim  = out_dim

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)


def _resnet50_feature_extractor(weights_path: Optional[str]) -> 'FeatureExtractor':
    model   = tvm.resnet50(weights=None)
    out_dim = model.fc.in_features  # 2048
    if weights_path and Path(weights_path).exists():
        state = torch.load(weights_path, map_location='cpu')
        if 'state_dict' in state: state = state['state_dict']
        state = {k.replace('module.', ''): v for k, v in state.items()}
        state = {k: v for k, v in state.items() if not k.startswith('fc.')}
        missing, unexpected = model.load_state_dict(state, strict=False)
        log.info(f'[RadImageNet ResNet-50] loaded. Missing: {len(missing)}, Unexpected: {len(unexpected)}')
    else:
        if weights_path:
            log.warning(f"[ResNet-50] Weights not found at '{weights_path}'. Falling back to ImageNet.")
        model = tvm.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        log.info('[ResNet-50] Loaded ImageNet V2 weights.')
    model.fc = nn.Identity()
    return FeatureExtractor(nn.Sequential(model), out_dim)


def _densenet121_feature_extractor(weights_path: Optional[str]) -> 'FeatureExtractor':
    class DenseNet121Features(nn.Module):
        def __init__(self, base):
            super().__init__()
            self.features = base.features
            self.pool     = nn.AdaptiveAvgPool2d((1, 1))
        def forward(self, x):
            x = self.features(x)
            x = F.relu(x, inplace=True)
            x = self.pool(x)
            return x.flatten(1)

    base    = tvm.densenet121(weights=None)
    out_dim = base.classifier.in_features  # 1024
    if weights_path and Path(weights_path).exists():
        state = torch.load(weights_path, map_location='cpu')
        if 'state_dict' in state: state = state['state_dict']
        state = {k.replace('module.', ''): v for k, v in state.items()
                 if not k.startswith('classifier.')}
        missing, unexpected = base.load_state_dict(state, strict=False)
        log.info(f'[RadImageNet DenseNet-121] loaded. Missing: {len(missing)}, Unexpected: {len(unexpected)}')
    else:
        if weights_path:
            log.warning(f"[DenseNet-121] Weights not found. Falling back to ImageNet.")
        base = tvm.densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)
        log.info('[DenseNet-121] Loaded ImageNet V1 weights.')
    return FeatureExtractor(DenseNet121Features(base), out_dim)


def _efficientnet_b2_feature_extractor() -> 'FeatureExtractor':
    model   = tvm.efficientnet_b2(weights=EfficientNet_B2_Weights.IMAGENET1K_V1)
    out_dim = model.classifier[1].in_features  # 1408
    model.classifier = nn.Identity()
    model.avgpool    = nn.AdaptiveAvgPool2d((1, 1))

    class EfficientWrapper(nn.Module):
        def __init__(self, base):
            super().__init__()
            self.base = base
        def forward(self, x):
            x = self.base.features(x)
            x = self.base.avgpool(x)
            return x.flatten(1)

    log.info('[EfficientNet-B2] Loaded ImageNet weights.')
    return FeatureExtractor(EfficientWrapper(model), out_dim)


def _openus_vit_b16_feature_extractor(weights_path: Optional[str]) -> 'FeatureExtractor':
    try:
        import timm
    except ImportError:
        raise ImportError('timm is required for OpenUS ViT backbone: pip install timm')
    model   = timm.create_model('vit_base_patch16_224', pretrained=False,
                                 img_size=CFG.IMAGE_SIZE, num_classes=0)
    out_dim = model.embed_dim  # 768
    if weights_path and Path(weights_path).exists():
        state = torch.load(weights_path, map_location='cpu')
        if 'model' in state: state = state['model']
        elif 'state_dict' in state: state = state['state_dict']
        state = {k.replace('module.', ''): v for k, v in state.items()}
        missing, unexpected = model.load_state_dict(state, strict=False)
        log.info(f'[OpenUS ViT-B/16] Loaded. Missing: {len(missing)}, Unexpected: {len(unexpected)}')
    else:
        log.warning('[OpenUS ViT-B/16] No weights file — random initialisation.')
    return FeatureExtractor(model, out_dim)


def _echoCare_stub_extractor() -> 'FeatureExtractor':
    log.warning('[EchoCare/US-JEPA] Weights not yet public. Using EfficientNet-B2 as stand-in.')
    return _efficientnet_b2_feature_extractor()


def build_backbone(name: str, weights_path: Optional[str] = None) -> 'FeatureExtractor':
    registry = {
        'radimagenet_resnet50':     lambda: _resnet50_feature_extractor(weights_path),
        'radimagenet_densenet121':  lambda: _densenet121_feature_extractor(weights_path),
        'openus_vit_b16':           lambda: _openus_vit_b16_feature_extractor(weights_path),
        'imagenet_efficientnet_b2': lambda: _efficientnet_b2_feature_extractor(),
        'imagenet_resnet50':        lambda: _resnet50_feature_extractor(None),
        'echocare':                 lambda: _echoCare_stub_extractor(),
        'us_jepa':                  lambda: _echoCare_stub_extractor(),
    }
    if name not in registry:
        raise ValueError(f"Unknown backbone '{name}'. Available: {list(registry.keys())}")
    extractor = registry[name]()
    log.info(f"Backbone '{name}' ready. Output dim: {extractor.out_dim}")
    return extractor


# Smoke-test
if __name__ == '__main__':
    _test = build_backbone('imagenet_resnet50')
    _x    = torch.randn(2, 3, CFG.IMAGE_SIZE, CFG.IMAGE_SIZE)
    _out  = _test(_x)
    assert _out.shape == (2, _test.out_dim)
    log.info(f'Backbone smoke-test passed. Output shape: {_out.shape}')
    del _test, _x, _out

## Cell 3 — Dataset Class

### Key design decisions

**Ordinal encoding** — A raw score `s ∈ {0,1,2,3}` is converted to a length-3
binary vector of cumulative targets:

| Score | P(≥1) | P(≥2) | P(≥3) | Vector |
|-------|-------|-------|-------|--------|
| 0     | 0     | 0     | 0     | [0,0,0] |
| 1     | 1     | 0     | 0     | [1,0,0] |
| 2     | 1     | 1     | 0     | [1,1,0] |
| 3     | 1     | 1     | 1     | [1,1,1] |

**Masking** — For each task, a scalar mask (0 or 1) is returned alongside the
target. A mask of 0 means the sample should not contribute to this task's loss
because the label is NaN or the modality makes the task impossible.

**Augmentation** — Ultrasound-safe transforms only. No hue jitter on Doppler,
no large rotations, no horizontal flip unless laterality is normalised.

In [ ]:
_IMAGENET_MEAN = (0.485, 0.456, 0.406)
_IMAGENET_STD  = (0.229, 0.224, 0.225)


def build_train_transforms(image_size: int, is_doppler: bool = False) -> A.Compose:
    colour_transforms = [] if is_doppler else [
        A.RandomBrightnessContrast(brightness_limit=0.20, contrast_limit=0.20, p=0.5),
        A.CLAHE(clip_limit=(1.0, 4.0), tile_grid_size=(8, 8), p=0.4),
    ]
    return A.Compose([
        A.Resize(image_size, image_size),
        A.Rotate(limit=12, border_mode=0, p=0.5),
        A.ElasticTransform(alpha=30, sigma=5, p=0.3),
        A.RandomResizedCrop(size=(image_size, image_size), scale=(0.85, 1.0), ratio=(0.9, 1.1), p=0.4),
        *colour_transforms,
        A.GaussNoise(var_limit=(5.0, 20.0), p=0.4),
        A.CoarseDropout(max_holes=4, max_height=image_size // 10,
                        max_width=image_size // 10, fill_value=0, p=0.3),
        A.Normalize(mean=_IMAGENET_MEAN, std=_IMAGENET_STD),
        ToTensorV2(),
    ])


def build_val_transforms(image_size: int) -> A.Compose:
    return A.Compose([
        A.Resize(image_size, image_size),
        A.Normalize(mean=_IMAGENET_MEAN, std=_IMAGENET_STD),
        ToTensorV2(),
    ])


def score_to_cumulative_target(score: float, n_thresholds: int = 3) -> torch.Tensor:
    score  = int(score)
    target = torch.zeros(n_thresholds, dtype=torch.float32)
    target[:score] = 1.0
    return target


class MSKUltrasoundDataset(Dataset):
    BRANCH_MODALITY = {'structural': 0, 'vascular': 1}

    def __init__(self, dataframe, image_dir, config, transform=None, is_train=True):
        self.df        = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.cfg       = config
        self.is_train  = is_train
        self.transform = transform or (
            build_train_transforms(config.IMAGE_SIZE) if is_train
            else build_val_transforms(config.IMAGE_SIZE)
        )
        required_cols = (
            [config.ECO_ID_COL, config.MODALITY_COL, config.JOINT_TYPE_COL]
            + [t['col'] for t in config.TASKS.values()]
        )
        missing = [c for c in required_cols if c not in self.df.columns]
        if missing:
            raise ValueError(f'DataFrame missing required columns: {missing}')
        log.info(f"Dataset ready: {len(self.df)} samples, {'train' if is_train else 'val/test'} mode.")

    def __len__(self):
        return len(self.df)

    def _load_image(self, eco_id: str, modality_id: int) -> np.ndarray:
        img_path = self.image_dir / f'{eco_id}.png'
        if not img_path.exists():
            raise FileNotFoundError(f'Image not found: {img_path}')
        img = Image.open(img_path)
        if self.cfg.GRAYSCALE_TO_RGB:
            img = img.convert('RGB')
        else:
            img = np.array(img)
            if img.ndim == 2:
                img = np.stack([img] * 3, axis=-1)
        return np.array(img)

    def _build_masks(self, row, modality_id):
        masks = {}
        for task_name, task_cfg in self.cfg.TASKS.items():
            col          = task_cfg['col']
            required_mod = self.BRANCH_MODALITY[task_cfg['branch']]
            masks[task_name] = 0.0 if (pd.isna(row[col]) or modality_id != required_mod) else 1.0
        return masks

    def __getitem__(self, idx):
        row         = self.df.iloc[idx]
        eco_id      = str(row[self.cfg.ECO_ID_COL])
        modality_id = int(row[self.cfg.MODALITY_COL])
        joint_id    = int(row[self.cfg.JOINT_TYPE_COL])
        img_np      = self._load_image(eco_id, modality_id)
        img_t       = self.transform(image=img_np)['image']
        masks       = self._build_masks(row, modality_id)
        targets     = {}
        for task_name, task_cfg in self.cfg.TASKS.items():
            if masks[task_name] == 1.0:
                targets[task_name] = score_to_cumulative_target(
                    float(row[task_cfg['col']]), self.cfg.ORDINAL_THRESHOLDS)
            else:
                targets[task_name] = torch.zeros(self.cfg.ORDINAL_THRESHOLDS, dtype=torch.float32)
        return {
            'image':       img_t,
            'modality_id': torch.tensor(modality_id, dtype=torch.long),
            'joint_id':    torch.tensor(joint_id,    dtype=torch.long),
            'targets':     targets,
            'masks':       {k: torch.tensor(v, dtype=torch.float32) for k, v in masks.items()},
            'eco_id':      eco_id,
        }


def hydra_collate(batch):
    images      = torch.stack([b['image']       for b in batch])
    modality_id = torch.stack([b['modality_id'] for b in batch])
    joint_id    = torch.stack([b['joint_id']    for b in batch])
    eco_ids     = [b['eco_id'] for b in batch]
    task_names  = list(batch[0]['targets'].keys())
    targets     = {t: torch.stack([b['targets'][t] for b in batch]) for t in task_names}
    masks       = {t: torch.stack([b['masks'][t]   for b in batch]) for t in task_names}
    return {'image': images, 'modality_id': modality_id, 'joint_id': joint_id,
            'targets': targets, 'masks': masks, 'eco_ids': eco_ids}


def _dataset_sanity_check(df_sample):
    log.info('=== Dataset Sanity Check ===')
    log.info(f'  DataFrame shape: {df_sample.shape}')
    for task_name, task_cfg in CFG.TASKS.items():
        nan_count = df_sample[task_cfg['col']].isna().sum()
        log.info(f"  Task '{task_name}': {nan_count}/{len(df_sample)} NaN")
    for score in range(4):
        t = score_to_cumulative_target(score)
        assert t.shape == (3,) and t.sum() == score
    log.info('  Ordinal encoding: OK')
    log.info('=== Sanity Check Passed ===')

# Uncomment to run after loading your DataFrame:
# _dataset_sanity_check(df)

## Cell 4 — The Hydra Model (`nn.Module`)

### Architecture overview
```
Image (B, 3, H, W)
     │
  Backbone  ──────────────────────────────────────────────── swappable
     │ (B, backbone_out_dim)
  Linear projection → (B, FEATURE_DIM)
     │
  GroupNorm + GELU
     │
  ┌─────────────────────────────────────────┐
  │  Metadata conditioning                  │
  │  modality_id → Embedding (B, 16)        │
  │  joint_id    → Embedding (B, 32)        │
  │  ↓ concat → (B, FEATURE_DIM + 48)       │
  └─────────────────────────────────────────┘
     │
  ┌──────────────────┐   ┌──────────────────┐
  │  Structural Head │   │  Vascular Head   │
  │  (B-Mode tasks)  │   │  (Doppler tasks) │
  │  eg_sinovial [3] │   │  pd_sinovial [3] │
  │  bone_erosion[3] │   │  ext_doppler [3] │
  │  ext_hypert. [3] │   │                  │
  └──────────────────┘   └──────────────────┘
```

**All heads always compute outputs.** Masking is done entirely in the loss function.

**Why GroupNorm over BatchNorm?** With small batches forced by GPU memory on 320px images,
BN running-mean estimates become unstable. GroupNorm has no batch-size dependency.

In [ ]:
class OrdinalTaskHead(nn.Module):
    """Single task head producing `n_thresholds` logits for ordinal regression."""
    def __init__(self, in_dim, n_thresholds=3, hidden_dim=128, dropout=0.3):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, n_thresholds),
        )
        nn.init.normal_(self.head[-1].weight, std=0.01)
        nn.init.zeros_(self.head[-1].bias)

    def forward(self, x):
        return self.head(x)


class ModalityBranch(nn.Module):
    """Shared sub-network for all tasks in one modality branch."""
    def __init__(self, in_dim, tasks, n_thresh=3, dropout=0.3):
        super().__init__()
        self.task_names = list(tasks.keys())
        self.branch_net = nn.Sequential(
            nn.Linear(in_dim, in_dim), nn.GELU(), nn.Dropout(dropout))
        self.task_heads = nn.ModuleDict({
            t: OrdinalTaskHead(in_dim, n_thresh, dropout=dropout)
            for t in self.task_names
        })

    def forward(self, x):
        shared = self.branch_net(x)
        return {t: self.task_heads[t](shared) for t in self.task_names}


class HydraModel(nn.Module):
    """Unified multi-task ordinal regression model for MSK ultrasound scoring."""
    def __init__(self, config: HydraConfig, backbone: FeatureExtractor):
        super().__init__()
        self.cfg      = config
        self.backbone = backbone

        self.feature_proj = nn.Sequential(
            nn.Linear(backbone.out_dim, config.FEATURE_DIM),
            nn.GroupNorm(32, config.FEATURE_DIM),
            nn.GELU(),
            nn.Dropout(config.DROPOUT_RATE),
        )
        self.modality_embedding = nn.Embedding(config.NUM_MODALITIES,  config.MODALITY_EMBED_DIM)
        self.joint_embedding    = nn.Embedding(config.NUM_JOINT_TYPES,  config.JOINT_EMBED_DIM)

        conditioned_dim = (config.FEATURE_DIM + config.MODALITY_EMBED_DIM + config.JOINT_EMBED_DIM)
        structural_tasks = {k: v for k, v in config.TASKS.items() if v['branch'] == 'structural'}
        vascular_tasks   = {k: v for k, v in config.TASKS.items() if v['branch'] == 'vascular'}

        self.structural_branch = ModalityBranch(
            conditioned_dim, structural_tasks, config.ORDINAL_THRESHOLDS, config.DROPOUT_RATE)
        self.vascular_branch   = ModalityBranch(
            conditioned_dim, vascular_tasks,   config.ORDINAL_THRESHOLDS, config.DROPOUT_RATE)

        total_params    = sum(p.numel() for p in self.parameters())
        backbone_params = sum(p.numel() for p in self.backbone.parameters())
        log.info(f'[HydraModel] Total: {total_params/1e6:.2f}M | '
                 f'Backbone: {backbone_params/1e6:.2f}M | '
                 f'Heads: {(total_params-backbone_params)/1e6:.2f}M')

    def forward(self, image, modality_id, joint_id):
        feats       = self.backbone(image)
        feats       = self.feature_proj(feats)
        mod_emb     = self.modality_embedding(modality_id)
        joint_emb   = self.joint_embedding(joint_id)
        conditioned = torch.cat([feats, mod_emb, joint_emb], dim=-1)
        return {**self.structural_branch(conditioned), **self.vascular_branch(conditioned)}

    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False
        log.info('Backbone frozen.')

    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = True
        log.info('Backbone unfrozen.')

    def get_param_groups(self, base_lr, backbone_lr_mult):
        backbone_ids    = set(id(p) for p in self.backbone.parameters())
        backbone_params = [p for p in self.parameters() if     id(p) in backbone_ids]
        head_params     = [p for p in self.parameters() if not id(p) in backbone_ids]
        return [{'params': head_params,     'lr': base_lr},
                {'params': backbone_params, 'lr': base_lr * backbone_lr_mult}]


def build_hydra_model(cfg: HydraConfig) -> HydraModel:
    backbone = build_backbone(cfg.BACKBONE_NAME, cfg.BACKBONE_WEIGHTS_PATH)
    return HydraModel(cfg, backbone).to(DEVICE)


def _model_smoke_test(cfg):
    log.info('=== Model Smoke-Test ===')
    model = build_hydra_model(cfg)
    model.eval()
    B = 4
    with torch.no_grad():
        outputs = model(
            torch.randn(B, 3, cfg.IMAGE_SIZE, cfg.IMAGE_SIZE).to(DEVICE),
            torch.randint(0, cfg.NUM_MODALITIES,  (B,)).to(DEVICE),
            torch.randint(0, cfg.NUM_JOINT_TYPES, (B,)).to(DEVICE),
        )
    for task_name, logits in outputs.items():
        assert logits.shape == (B, cfg.ORDINAL_THRESHOLDS)
        log.info(f"  Task '{task_name}': shape={logits.shape} OK")
    log.info('=== Smoke-Test Passed ===')
    del model
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

_model_smoke_test(CFG)

## Cell 5 — Masked Multi-Task Loss

### Critical implementation notes

The masked BCE loss works by element-wise multiplying the per-sample loss by its
mask *before* reduction — safer than zeroing out entries post-hoc.

**Normalisation strategy** — Normalised by the actual number of *labeled samples*
across all tasks in the batch, not the batch size.

**Task weighting** — Each task's loss is scaled by its configured weight before
summing. Use higher weights for clinically important or data-sparse tasks.

In [ ]:
class MaskedOrdinalLoss(nn.Module):
    """Multi-task masked ordinal BCE loss."""
    def __init__(self, task_weights: Dict[str, float]):
        super().__init__()
        self.task_weights = task_weights

    def forward(self, predictions, targets, masks):
        total_loss        = torch.tensor(0.0, device=next(iter(predictions.values())).device)
        per_task_loss     = {}
        total_valid_count = 0

        for task_name, pred_logits in predictions.items():
            target  = targets[task_name]
            mask    = masks[task_name]
            n_valid = mask.sum()

            if n_valid == 0:
                per_task_loss[task_name] = torch.tensor(0.0, device=pred_logits.device)
                continue

            element_loss  = F.binary_cross_entropy_with_logits(pred_logits, target, reduction='none')
            sample_loss   = element_loss.mean(dim=-1)
            masked_loss   = (sample_loss * mask).sum() / n_valid
            w             = self.task_weights.get(task_name, 1.0)
            total_loss    = total_loss + w * masked_loss
            per_task_loss[task_name] = masked_loss.detach()
            total_valid_count += n_valid.item()

        return total_loss, per_task_loss


def _loss_unit_tests():
    log.info('=== Loss Unit Tests ===')
    task_names = list(CFG.TASKS.keys())
    n_thresh   = CFG.ORDINAL_THRESHOLDS
    B          = 6
    loss_fn    = MaskedOrdinalLoss({t: 1.0 for t in task_names})

    # Test 1: all masks = 0 → loss must be 0
    preds   = {t: torch.zeros(B, n_thresh) for t in task_names}
    targets = {t: torch.zeros(B, n_thresh) for t in task_names}
    masks   = {t: torch.zeros(B)           for t in task_names}
    loss, _ = loss_fn(preds, targets, masks)
    assert loss.item() == 0.0
    log.info('  Test 1 (all masks=0): OK')

    # Test 2: perfect predictions → near-zero loss
    perfect_logits = torch.tensor([[10., 10., -10.]] * B)
    perfect_target = torch.tensor([[1.,  1.,  0.]] * B)
    preds   = {t: perfect_logits.clone() for t in task_names}
    targets = {t: perfect_target.clone() for t in task_names}
    masks   = {t: torch.ones(B)          for t in task_names}
    loss, _ = loss_fn(preds, targets, masks)
    assert loss.item() < 0.01
    log.info('  Test 2 (perfect preds): OK')

    # Test 3: masked task must not affect gradient
    t0            = task_names[0]
    preds_bad     = {t: torch.zeros(B, n_thresh) for t in task_names}
    preds_bad[t0] = torch.ones(B, n_thresh) * 1000
    masks_no_t0   = {t: (torch.zeros(B) if t == t0 else torch.ones(B)) for t in task_names}
    _, per_task   = loss_fn(preds_bad, targets, masks_no_t0)
    assert per_task[t0].item() == 0.0
    log.info(f"  Test 3 (task '{t0}' masked out): OK")

    # Test 4: NaN targets with zero mask must not produce NaN loss
    nan_targets  = {t: torch.full((B, n_thresh), float('nan')) for t in task_names}
    all_zero_msk = {t: torch.zeros(B) for t in task_names}
    preds_any    = {t: torch.randn(B, n_thresh) for t in task_names}
    loss_nan, _  = loss_fn(preds_any, nan_targets, all_zero_msk)
    assert not torch.isnan(loss_nan)
    log.info('  Test 4 (NaN targets, zero mask): OK')
    log.info('=== All Loss Unit Tests Passed ===')

_loss_unit_tests()

## Cell 6 — Metrics (QWK + MAE)

In [ ]:
def logits_to_ordinal_score(logits: torch.Tensor) -> torch.Tensor:
    """Convert (B, n_thresh) logits → (B,) predicted ordinal scores."""
    probs = torch.sigmoid(logits)
    return (probs > 0.5).sum(dim=-1).long()


class MetricAccumulator:
    """Accumulates predictions over an epoch, then computes QWK and MAE."""
    def __init__(self, task_names):
        self.task_names = task_names
        self._preds = {t: [] for t in task_names}
        self._trues = {t: [] for t in task_names}

    def update(self, logits, targets, masks):
        for task_name in self.task_names:
            mask = masks[task_name].bool().cpu()
            if mask.sum() == 0: continue
            pred_scores = logits_to_ordinal_score(logits[task_name].detach().cpu())[mask].numpy()
            true_scores = targets[task_name].cpu()[mask].sum(dim=-1).long().numpy()
            self._preds[task_name].append(pred_scores)
            self._trues[task_name].append(true_scores)

    def compute(self):
        results = {}
        for task_name in self.task_names:
            if not self._preds[task_name]:
                results[task_name] = {'qwk': float('nan'), 'mae': float('nan'), 'n': 0}
                continue
            preds = np.concatenate(self._preds[task_name])
            trues = np.concatenate(self._trues[task_name])
            if len(np.unique(trues)) < 2:
                qwk = float('nan')
            else:
                try:
                    qwk = cohen_kappa_score(trues, preds, weights='quadratic', labels=[0, 1, 2, 3])
                except ValueError:
                    qwk = float('nan')
            mae = float(np.mean(np.abs(preds - trues)))
            results[task_name] = {'qwk': qwk, 'mae': mae, 'n': len(preds)}
        return results

    def reset(self):
        self._preds = {t: [] for t in self.task_names}
        self._trues = {t: [] for t in self.task_names}

    def mean_qwk(self):
        metrics = self.compute()
        qwks    = [v['qwk'] for v in metrics.values() if not np.isnan(v['qwk'])]
        return float(np.mean(qwks)) if qwks else float('nan')

## Cell 7 — Training & Validation Loop Functions

In [ ]:
from tqdm.auto import tqdm


def train_one_epoch(model, loader, optimizer, loss_fn, scaler, cfg, epoch):
    model.train()
    task_names     = list(cfg.TASKS.keys())
    epoch_loss     = 0.0
    task_loss_sums = {t: 0.0 for t in task_names}
    n_batches      = 0

    pbar = tqdm(loader, desc=f'Epoch {epoch:03d} [TRAIN]', leave=False)
    for batch in pbar:
        images      = batch['image'].to(DEVICE, non_blocking=True)
        modality_id = batch['modality_id'].to(DEVICE, non_blocking=True)
        joint_id    = batch['joint_id'].to(DEVICE, non_blocking=True)
        targets     = {t: v.to(DEVICE, non_blocking=True) for t, v in batch['targets'].items()}
        masks       = {t: v.to(DEVICE, non_blocking=True) for t, v in batch['masks'].items()}

        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type=DEVICE.type, enabled=cfg.USE_AMP and DEVICE.type == 'cuda'):
            predictions = model(images, modality_id, joint_id)
            loss, per_task = loss_fn(predictions, targets, masks)

        if cfg.USE_AMP and DEVICE.type == 'cuda':
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP_NORM)
            optimizer.step()

        epoch_loss += loss.item()
        for t in task_names:
            task_loss_sums[t] += per_task.get(t, torch.tensor(0.0)).item()
        n_batches += 1
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    return epoch_loss / max(n_batches, 1), {t: task_loss_sums[t] / max(n_batches, 1) for t in task_names}


def validate_one_epoch(model, loader, loss_fn, cfg, epoch):
    model.eval()
    task_names  = list(cfg.TASKS.keys())
    epoch_loss  = 0.0
    n_batches   = 0
    accumulator = MetricAccumulator(task_names)

    pbar = tqdm(loader, desc=f'Epoch {epoch:03d} [VAL] ', leave=False)
    with torch.no_grad():
        for batch in pbar:
            images      = batch['image'].to(DEVICE, non_blocking=True)
            modality_id = batch['modality_id'].to(DEVICE, non_blocking=True)
            joint_id    = batch['joint_id'].to(DEVICE, non_blocking=True)
            targets     = {t: v.to(DEVICE, non_blocking=True) for t, v in batch['targets'].items()}
            masks       = {t: v.to(DEVICE, non_blocking=True) for t, v in batch['masks'].items()}

            with autocast(device_type=DEVICE.type, enabled=cfg.USE_AMP and DEVICE.type == 'cuda'):
                predictions = model(images, modality_id, joint_id)
                loss, _     = loss_fn(predictions, targets, masks)

            epoch_loss += loss.item()
            n_batches  += 1
            accumulator.update(
                {t: predictions[t].cpu() for t in task_names},
                {t: targets[t].cpu()     for t in task_names},
                {t: masks[t].cpu()       for t in task_names},
            )

    return epoch_loss / max(n_batches, 1), accumulator.compute()


def save_checkpoint(model, optimizer, scheduler, epoch, fold, val_loss, mean_qwk, cfg, filepath):
    torch.save({
        'epoch': epoch, 'fold': fold,
        'model_state': model.state_dict(),
        'optim_state': optimizer.state_dict(),
        'sched_state': scheduler.state_dict() if scheduler else None,
        'val_loss': val_loss, 'mean_qwk': mean_qwk,
        'config': asdict(cfg),
    }, filepath)


def load_checkpoint(filepath, model, optimizer=None, scheduler=None):
    ckpt = torch.load(filepath, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'])
    if optimizer and 'optim_state' in ckpt: optimizer.load_state_dict(ckpt['optim_state'])
    if scheduler and 'sched_state' in ckpt and ckpt['sched_state']:
        scheduler.load_state_dict(ckpt['sched_state'])
    log.info(f"Loaded checkpoint epoch {ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f}, mean_qwk={ckpt['mean_qwk']:.4f}")
    return ckpt['epoch']


def log_epoch_results(epoch, fold, train_loss, val_loss, metrics):
    qwk_str  = '  '.join(f"{t[:12]}: QWK={v['qwk']:.3f} MAE={v['mae']:.3f} (n={v['n']})" for t, v in metrics.items())
    mean_qwk = np.nanmean([v['qwk'] for v in metrics.values()])
    log.info(f'Fold {fold} | Epoch {epoch:03d} | Train Loss={train_loss:.4f} | Val Loss={val_loss:.4f} | Mean QWK={mean_qwk:.3f}')
    log.info(f'  ↳ {qwk_str}')

## Cell 8 — Cross-Validation Main Loop

### CV strategy
- **Hospital B is locked out** before any splitting occurs — used only for the final blind evaluation in Cell 9.
- **GroupKFold by `patient_id`** ensures all images from the same patient land in the same fold.
- **Stratification key** — composite key from `hospital_id` + primary task ordinal score.
- **Differential LR** — Backbone uses `BACKBONE_LR_MULTIPLIER × base_LR`.
- **Backbone freeze warmup** — For the first `FREEZE_BACKBONE_EPOCHS` epochs, only heads are updated.

In [ ]:
# ── Load cleaned DataFrame ───────────────────────────────────────────────────
df_all = pd.read_csv(CFG.LABELS_CSV)
log.info(f'Loaded DataFrame: {df_all.shape}')

# ── Separate Hospital B (blind test set) ─────────────────────────────────────
if CFG.HOSPITAL_B_ECO_IDS:
    df_test_B  = df_all[df_all[CFG.ECO_ID_COL].isin(CFG.HOSPITAL_B_ECO_IDS)].copy()
    df_train_A = df_all[~df_all[CFG.ECO_ID_COL].isin(CFG.HOSPITAL_B_ECO_IDS)].copy()
else:
    log.warning('CFG.HOSPITAL_B_ECO_IDS is empty. All data treated as Hospital A.')
    df_test_B  = pd.DataFrame()
    df_train_A = df_all.copy()

log.info(f'Hospital A (training pool): {len(df_train_A)} samples')
log.info(f'Hospital B (blind test):    {len(df_test_B)} samples')
df_train_A = df_train_A.reset_index(drop=True)

# ── Build stratification key & groups ────────────────────────────────────────
if CFG.PATIENT_ID_COL and CFG.PATIENT_ID_COL in df_train_A.columns:
    groups = df_train_A[CFG.PATIENT_ID_COL].values
    log.info(f"Using '{CFG.PATIENT_ID_COL}' for group-aware CV. Unique patients: {len(np.unique(groups))}")
else:
    log.warning(f"Column '{CFG.PATIENT_ID_COL}' not found. Falling back to eco_id as group.")
    groups = df_train_A[CFG.ECO_ID_COL].values

primary_task_col = list(CFG.TASKS.values())[0]['col']
strat_score      = df_train_A[primary_task_col].fillna(-1).astype(int)
strat_key        = (
    df_train_A[CFG.HOSPITAL_COL].astype(str) + '_' + strat_score.astype(str)
    if CFG.HOSPITAL_COL in df_train_A.columns else strat_score.astype(str)
).values

splitter  = StratifiedGroupKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.RANDOM_SEED)
X_dummy   = np.zeros(len(df_train_A))
cv_splits = list(splitter.split(X_dummy, strat_key, groups=groups))
log.info(f'CV splits created: {CFG.N_FOLDS} folds')

cv_results = []

for fold_idx, (train_idx, val_idx) in enumerate(cv_splits):
    log.info('=' * 70)
    log.info(f'FOLD {fold_idx + 1} / {CFG.N_FOLDS}')
    log.info('=' * 70)
    seed_everything(CFG.RANDOM_SEED + fold_idx)

    df_fold_train = df_train_A.iloc[train_idx].reset_index(drop=True)
    df_fold_val   = df_train_A.iloc[val_idx].reset_index(drop=True)

    train_dataset = MSKUltrasoundDataset(df_fold_train, CFG.IMAGE_DIR, CFG, is_train=True)
    val_dataset   = MSKUltrasoundDataset(df_fold_val,   CFG.IMAGE_DIR, CFG, is_train=False)

    primary_col    = list(CFG.TASKS.values())[0]['col']
    score_counts   = df_fold_train[primary_col].fillna(0).value_counts()
    score_weight   = 1.0 / score_counts.clip(lower=1)
    sample_weights = df_fold_train[primary_col].fillna(0).map(score_weight).fillna(1.0)
    sampler        = WeightedRandomSampler(
        weights=torch.tensor(sample_weights.values, dtype=torch.double),
        num_samples=len(train_dataset), replacement=True,
    )

    train_loader = DataLoader(
        train_dataset, batch_size=CFG.BATCH_SIZE, sampler=sampler,
        num_workers=CFG.NUM_WORKERS, pin_memory=CFG.PIN_MEMORY,
        collate_fn=hydra_collate, drop_last=True)
    val_loader   = DataLoader(
        val_dataset, batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
        num_workers=CFG.NUM_WORKERS, pin_memory=CFG.PIN_MEMORY,
        collate_fn=hydra_collate)

    model     = build_hydra_model(CFG)
    loss_fn   = MaskedOrdinalLoss(task_weights={t: v['weight'] for t, v in CFG.TASKS.items()})
    optimizer = torch.optim.AdamW(
        model.get_param_groups(CFG.LEARNING_RATE, CFG.BACKBONE_LR_MULTIPLIER),
        weight_decay=CFG.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=CFG.LR_T0, T_mult=CFG.LR_T_MULT, eta_min=1e-6)
    scaler    = GradScaler(enabled=CFG.USE_AMP and DEVICE.type == 'cuda')

    if CFG.FREEZE_BACKBONE_EPOCHS > 0:
        model.freeze_backbone()

    best_mean_qwk     = -float('inf')
    best_val_loss     = float('inf')
    epochs_no_improve = 0
    fold_history      = {'train_loss': [], 'val_loss': [], 'mean_qwk': [], 'per_task_metrics': []}
    ckpt_path         = os.path.join(CFG.CHECKPOINT_DIR, f'fold{fold_idx}_best.pth')

    for epoch in range(1, CFG.NUM_EPOCHS + 1):
        if epoch == CFG.FREEZE_BACKBONE_EPOCHS + 1:
            model.unfreeze_backbone()

        train_loss, _         = train_one_epoch(model, train_loader, optimizer, loss_fn, scaler, CFG, epoch)
        val_loss, val_metrics = validate_one_epoch(model, val_loader, loss_fn, CFG, epoch)
        scheduler.step(epoch - 1)

        mean_qwk = float(np.nanmean([v['qwk'] for v in val_metrics.values()]))
        log_epoch_results(epoch, fold_idx + 1, train_loss, val_loss, val_metrics)
        fold_history['train_loss'].append(train_loss)
        fold_history['val_loss'].append(val_loss)
        fold_history['mean_qwk'].append(mean_qwk)
        fold_history['per_task_metrics'].append(val_metrics)

        if mean_qwk > best_mean_qwk:
            best_mean_qwk = mean_qwk
            best_val_loss = val_loss
            epochs_no_improve = 0
            save_checkpoint(model, optimizer, scheduler, epoch, fold_idx,
                            val_loss, mean_qwk, CFG, ckpt_path)
            log.info(f'  New best checkpoint: Mean QWK={mean_qwk:.4f} saved.')
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= CFG.EARLY_STOP_PATIENCE:
            log.info(f'  Early stopping at epoch {epoch}.')
            break

    log.info(f'Fold {fold_idx + 1} done. Best Mean QWK: {best_mean_qwk:.4f}')
    cv_results.append({'fold': fold_idx, 'best_qwk': best_mean_qwk, 'best_val_loss': best_val_loss,
                        'history': fold_history, 'ckpt_path': ckpt_path})
    del model, optimizer, scheduler, scaler, train_loader, val_loader, train_dataset, val_dataset
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

# ── CV Summary ────────────────────────────────────────────────────────────────
qwks = [r['best_qwk'] for r in cv_results]
log.info(f'CV Mean QWK: {np.mean(qwks):.4f} ± {np.std(qwks):.4f}')
best_fold_idx = int(np.argmax(qwks))
log.info(f'Best fold: {best_fold_idx}  checkpoint: {cv_results[best_fold_idx]["ckpt_path"]}')

## Cell 9 — Evaluation: Training Curves, Confusion Matrices & Hospital B Test

### Hospital B evaluation protocol

The Hospital B images are evaluated **once**, using the best-fold checkpoint,
after all hyperparameter decisions and fold selection are finalised.

⚠️ **Do not look at Hospital B metrics to make any architectural or
hyperparameter decisions.** If you peek and adjust, the blind test is invalidated.

In [ ]:
def plot_training_curves(cv_results, save_dir):
    n_folds = len(cv_results)
    fig, axes = plt.subplots(2, n_folds, figsize=(5 * n_folds, 8))
    if n_folds == 1: axes = axes.reshape(2, 1)

    for fold_idx, result in enumerate(cv_results):
        hist     = result['history']
        epochs_r = range(1, len(hist['train_loss']) + 1)

        ax = axes[0, fold_idx]
        ax.plot(epochs_r, hist['train_loss'], label='Train', linewidth=1.5)
        ax.plot(epochs_r, hist['val_loss'],   label='Val',   linewidth=1.5)
        ax.set_title(f'Fold {fold_idx + 1} — Loss')
        ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.legend(); ax.grid(alpha=0.3)

        ax = axes[1, fold_idx]
        ax.plot(epochs_r, hist['mean_qwk'], color='green', linewidth=1.5)
        ax.axhline(result['best_qwk'], color='red', linestyle='--',
                   label=f"Best QWK={result['best_qwk']:.3f}")
        ax.set_title(f'Fold {fold_idx + 1} — Mean QWK')
        ax.set_xlabel('Epoch'); ax.set_ylabel('QWK'); ax.set_ylim(-0.1, 1.0)
        ax.legend(); ax.grid(alpha=0.3)

    plt.tight_layout()
    out_path = os.path.join(save_dir, 'training_curves.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    log.info(f'Training curves saved to {out_path}')

plot_training_curves(cv_results, CFG.RESULTS_DIR)


def plot_per_task_qwk(cv_results, task_names, save_dir):
    task_qwks = {t: [] for t in task_names}
    for result in cv_results:
        last_metrics = result['history']['per_task_metrics'][-1]
        for t in task_names:
            task_qwks[t].append(last_metrics.get(t, {}).get('qwk', float('nan')))

    means = [np.nanmean(task_qwks[t]) for t in task_names]
    stds  = [np.nanstd(task_qwks[t])  for t in task_names]

    fig, ax = plt.subplots(figsize=(max(8, len(task_names) * 1.5), 5))
    bars = ax.bar(task_names, means, yerr=stds, capsize=5,
                  color=sns.color_palette('muted', len(task_names)),
                  edgecolor='black', linewidth=0.7)
    ax.set_ylim(0, 1.0); ax.set_ylabel('Quadratic Weighted Kappa')
    ax.set_title('Per-Task QWK — Mean ± Std Across Folds')
    ax.axhline(0.4, color='orange', linestyle='--', alpha=0.7, label='Fair (0.4)')
    ax.axhline(0.6, color='green',  linestyle='--', alpha=0.7, label='Good (0.6)')
    ax.legend(loc='lower right'); ax.tick_params(axis='x', rotation=20); ax.grid(axis='y', alpha=0.3)
    for bar, mean, std in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 0.01,
                f'{mean:.3f}', ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    out_path = os.path.join(CFG.RESULTS_DIR, 'per_task_qwk.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()

plot_per_task_qwk(cv_results, list(CFG.TASKS.keys()), CFG.RESULTS_DIR)


def evaluate_hospital_b(df_test, cfg, ckpt_path):
    if df_test is None or len(df_test) == 0:
        log.warning('Hospital B test set is empty — skipping blind evaluation.')
        return {}

    log.info('=' * 70)
    log.info('HOSPITAL B — BLIND TEST EVALUATION (call this ONCE only)')
    log.info('=' * 70)

    test_loader = DataLoader(
        MSKUltrasoundDataset(df_test, cfg.IMAGE_DIR, cfg, is_train=False),
        batch_size=cfg.BATCH_SIZE, shuffle=False,
        num_workers=cfg.NUM_WORKERS, collate_fn=hydra_collate,
    )
    model = build_hydra_model(cfg)
    load_checkpoint(ckpt_path, model)
    model.eval()

    task_names  = list(cfg.TASKS.keys())
    accumulator = MetricAccumulator(task_names)

    with torch.no_grad():
        for batch in tqdm(test_loader, desc='Hospital B inference'):
            images      = batch['image'].to(DEVICE)
            modality_id = batch['modality_id'].to(DEVICE)
            joint_id    = batch['joint_id'].to(DEVICE)
            targets     = {t: v.to(DEVICE) for t, v in batch['targets'].items()}
            masks       = {t: v.to(DEVICE) for t, v in batch['masks'].items()}
            predictions = model(images, modality_id, joint_id)
            accumulator.update(
                {t: predictions[t].cpu() for t in task_names},
                {t: targets[t].cpu()     for t in task_names},
                {t: masks[t].cpu()       for t in task_names},
            )

    metrics = accumulator.compute()
    log.info('Hospital B Results:')
    for task_name, m in metrics.items():
        log.info(f'  {task_name:25s} QWK={m["qwk"]:.4f}  MAE={m["mae"]:.4f}  n={m["n"]}')
    del model
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    return metrics

# Uncomment ONLY when all modelling decisions are final:
# hospital_b_metrics = evaluate_hospital_b(
#     df_test_B, CFG, cv_results[best_fold_idx]['ckpt_path'])

# ── CV Result Table ───────────────────────────────────────────────────────────
task_names   = list(CFG.TASKS.keys())
summary_rows = []
for result in cv_results:
    last_metrics = result['history']['per_task_metrics'][-1]
    row = {'fold': result['fold'], 'best_mean_qwk': result['best_qwk']}
    for t in task_names:
        m = last_metrics.get(t, {})
        row[f'{t}_qwk'] = m.get('qwk', float('nan'))
        row[f'{t}_mae'] = m.get('mae', float('nan'))
    summary_rows.append(row)

summary_df   = pd.DataFrame(summary_rows)
log.info('\n' + summary_df.to_string(index=False, float_format='{:.4f}'.format))
summary_path = os.path.join(CFG.RESULTS_DIR, 'cv_summary.csv')
summary_df.to_csv(summary_path, index=False)
log.info(f'CV summary saved to {summary_path}')